# Grad-CAM Faithfulness — Colab Entry Point

Run cells top to bottom. Code is pulled from GitHub each session (ephemeral);
checkpoints and results are saved to Google Drive (persistent across disconnects).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/Chahnapatel09/gradcam-faithfulness-medical.git"
REPO_DIR = "/content/gradcam-faithfulness-medical"
DRIVE_DIR = "/content/drive/MyDrive/gradcam-faithfulness-medical"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import sys
sys.path.append(f"{REPO_DIR}/src")

os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)

## Train

Checkpoints/results are written to Drive, so re-running this cell after a
disconnect resumes from the last saved epoch instead of starting over.

In [ ]:
from train import train

model = train(
    data_dir=f"{DRIVE_DIR}/data",
    checkpoint_dir=f"{DRIVE_DIR}/checkpoints",
    results_dir=f"{DRIVE_DIR}/results",
    epochs=15,
    batch_size=32,
    lr=1e-4,
)

## Check test-set misclassification count

Diagnostic only — val accuracy was ~99%, so before committing to the
correct-vs-incorrect faithfulness split we need to know how many actual
misclassifications exist to work with on the test set.

In [ ]:
import torch
from sklearn.metrics import confusion_matrix, classification_report

from data import get_dataloaders
from model import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_, _, test_loader = get_dataloaders(data_dir=f"{DRIVE_DIR}/data", batch_size=32)

eval_model = build_model().to(device)
eval_model.load_state_dict(torch.load(f"{DRIVE_DIR}/checkpoints/best_model.pth", map_location=device))
eval_model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = eval_model(images)
        preds = outputs.argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.squeeze().tolist())

cm = confusion_matrix(all_labels, all_preds)
n_correct = sum(p == l for p, l in zip(all_preds, all_labels))
n_incorrect = len(all_labels) - n_correct

print(f"Test set size: {len(all_labels)}")
print(f"Correct: {n_correct} | Incorrect: {n_incorrect}")
print("\nConfusion matrix:\n", cm)
print("\n", classification_report(all_labels, all_preds, target_names=["normal", "pneumonia"]))

## Sanity check: visualize Grad-CAM / Grad-CAM++ / HiResCAM on one image

Quick visual check that heatmaps look reasonable before building the
masking/deletion pipeline on top of them.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pytorch_grad_cam.utils.image import show_cam_on_image

from gradcam_utils import compute_saliency

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

sample_images, sample_labels = next(iter(test_loader))
idx = 0
image_tensor = sample_images[idx:idx + 1].to(device)
true_label = sample_labels[idx].item()

eval_model.eval()
with torch.no_grad():
    pred_class = eval_model(image_tensor).argmax(1).item()

img_np = image_tensor[0].cpu().permute(1, 2, 0).numpy()
img_np = np.clip(img_np * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, method in zip(axes, ["gradcam", "gradcam++", "hirescam"]):
    saliency = compute_saliency(eval_model, image_tensor, target_class=pred_class, method=method)
    overlay = show_cam_on_image(img_np, saliency, use_rgb=True)
    ax.imshow(overlay)
    ax.set_title(method)
    ax.axis("off")

fig.suptitle(f"true={true_label} pred={pred_class}")
plt.show()